# Hybrid TPU + GDDR6-AiM Design-Space Sweep

Sweeps the **AiM-side GlobalBuffer size** (the per-PU buffer, default 2 KB) and
the **PU-array fanout** (default 16 × 16 = 256 PUs) on a hybrid TPU+AiM
architecture while keeping the TPU side unchanged. For each (buffer, fanout)
combination, we record:

- **Total chip area** (from `spec.arch.total_area`, summed over all components)
- **Workload energy** (mapper, EDP-best Pareto point)
- **Workload latency** (mapper, EDP-best Pareto point)

Workload: `gpt3_6.7B_kv_cache.yaml` (8192 prompt tokens, batch 1).

> **Note:** Each sweep point re-runs the mapper. Expect roughly 1–2 minutes per
> point; the full 5×5 sweep takes on the order of half an hour. Progress is
> printed; intermediate results are cached to `sweep_results.csv` so you can
> resume after an interrupt.

In [1]:
import contextlib
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import accelforge as af

EXAMPLES = Path("../examples")
ARCH = EXAMPLES / "arches" / "tpu_aim_hybrid_sweep.yaml"
WORKLOAD = EXAMPLES / "workloads" / "gpt3_6.7B_kv_cache.yaml"
RESULTS_CSV = Path("sweep_results.csv")

BATCH_SIZE = 1
N_TOKENS = 8192

# Sweep grid (5 x 5 = 25 points)
BUFFER_KBS = [1, 2, 4, 8, 16]            # AiM GlobalBuffer size in KB
PU_FANOUTS = [8, 12, 16, 24, 32]         # square side; total PUs = fanout^2

print(f"Arch: {ARCH.name}")
print(f"Workload: {WORKLOAD.name} (B={BATCH_SIZE}, M={N_TOKENS})")
print(f"Sweep: buffer ∈ {BUFFER_KBS} KB, fanout ∈ {PU_FANOUTS} "
      f"(PU counts {[f**2 for f in PU_FANOUTS]})")

Arch: tpu_aim_hybrid_sweep.yaml
Workload: gpt3_6.7B_kv_cache.yaml (B=1, M=8192)
Sweep: buffer ∈ [1, 2, 4, 8, 16] KB, fanout ∈ [8, 12, 16, 24, 32] (PU counts [64, 144, 256, 576, 1024])


## 1. Area sweep (fast)

Area depends only on the architecture, not the mapping, so we compute it for
every point first without running the mapper.

In [2]:
def compute_area(buffer_kb: int, pu_fanout: int) -> dict:
    """Return total + per-component areas for a given (buffer, fanout) point."""
    params = {
        "BATCH_SIZE": BATCH_SIZE,
        "N_TOKENS": N_TOKENS,
        "AIM_BUFFER_KB": buffer_kb,
        "AIM_PU_FANOUT": pu_fanout,
    }
    spec = af.Spec.from_yaml(str(ARCH), str(WORKLOAD), jinja_parse_data=params)
    spec = spec.calculate_component_area_energy_latency_leak()
    return {
        "total_area_m2": spec.arch.total_area,
        "per_component": dict(spec.arch.per_component_total_area),
    }


area_rows = []
for buf in BUFFER_KBS:
    for fan in PU_FANOUTS:
        a = compute_area(buf, fan)
        area_rows.append({
            "buffer_kb": buf,
            "pu_fanout": fan,
            "pu_count": fan * fan,
            "total_area_mm2": a["total_area_m2"] * 1e6,
            "aim_buffer_area_mm2": a["per_component"].get("GlobalBuffer", 0) * 1e6,
            "pu_mac_area_mm2": a["per_component"].get("PU_MAC", 0) * 1e6,
        })

area_df = pd.DataFrame(area_rows)
area_df

,buffer_kb,pu_fanout,pu_count,total_area_mm2,aim_buffer_area_mm2,pu_mac_area_mm2
0,1,8,64,209.316992,0.003392,0.76
1,1,12,144,210.266992,0.003392,1.71
2,1,16,256,211.596992,0.003392,3.04
3,1,24,576,215.396992,0.003392,6.84
4,1,32,1024,220.716992,0.003392,12.16
5,2,8,64,209.320475,0.006875,0.76
6,2,12,144,210.270475,0.006875,1.71
7,2,16,256,211.600475,0.006875,3.04
8,2,24,576,215.400475,0.006875,6.84
9,2,32,1024,220.720475,0.006875,12.16


## 2. Energy + latency sweep (slow)

This is the long cell. For each point we run `map_workload_to_arch`, select the
EDP-best point from the Pareto set, and record energy + latency. Results are
written to `sweep_results.csv` after each point so the sweep can be interrupted
and resumed.

In [ ]:
def pick_edp_best(energies, latencies):
    """Return (energy, latency) minimizing energy*latency over a Pareto set."""
    if not isinstance(energies, list):
        return energies, latencies
    idx = min(range(len(energies)), key=lambda i: energies[i] * latencies[i])
    return energies[idx], latencies[idx]


def evaluate_point(buffer_kb: int, pu_fanout: int) -> dict:
    params = {
        "BATCH_SIZE": BATCH_SIZE,
        "N_TOKENS": N_TOKENS,
        "AIM_BUFFER_KB": buffer_kb,
        "AIM_PU_FANOUT": pu_fanout,
    }
    spec = af.Spec.from_yaml(str(ARCH), str(WORKLOAD), jinja_parse_data=params)
    spec.mapper.metrics = af.Metrics.LATENCY | af.Metrics.ENERGY
    # Silence mapper's stdout noise
    with contextlib.redirect_stdout(io.StringIO()):
        mapping = spec.map_workload_to_arch(print_progress=False)
    energy, latency = pick_edp_best(mapping.energy(), mapping.latency())
    return {"energy_j": energy, "latency_s": latency}


# Resume from CSV if it exists
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    done = {(int(r.buffer_kb), int(r.pu_fanout)) for r in results_df.itertuples()}
    print(f"Resuming: {len(done)}/{len(BUFFER_KBS)*len(PU_FANOUTS)} points already cached")
else:
    results_df = pd.DataFrame(columns=["buffer_kb", "pu_fanout", "energy_j", "latency_s"])
    done = set()

grid = [(b, f) for b in BUFFER_KBS for f in PU_FANOUTS]
total = len(grid)
for i, (buf, fan) in enumerate(grid, 1):
    if (buf, fan) in done:
        continue
    t0 = time.time()
    try:
        r = evaluate_point(buf, fan)
        row = {"buffer_kb": buf, "pu_fanout": fan, **r}
        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
        results_df.to_csv(RESULTS_CSV, index=False)
        dt = time.time() - t0
        print(f"[{i:2d}/{total}] buf={buf:>2}KB fan={fan:>2} "
              f"-> E={r['energy_j']:.3e} J, L={r['latency_s']:.3e} s  ({dt:.1f}s)")
    except Exception as e:
        print(f"[{i:2d}/{total}] buf={buf} fan={fan} FAILED: {e}")

results_df

## 3. Merge area + energy + latency and derive EDP

In [ ]:
df = results_df.merge(area_df, on=["buffer_kb", "pu_fanout"])
df["edp"] = df["energy_j"] * df["latency_s"]
df = df.sort_values(["buffer_kb", "pu_fanout"]).reset_index(drop=True)
df

## 4. Heatmaps

Rows = buffer size, columns = PU fanout. Lower is better for all four plots.

In [ ]:
def heat(ax, metric, title, cmap="viridis", log=False):
    pivot = df.pivot(index="buffer_kb", columns="pu_fanout", values=metric)
    data = pivot.values
    if log:
        data = np.log10(data)
    im = ax.imshow(data, aspect="auto", cmap=cmap, origin="lower")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{c}\n({c*c} PUs)" for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{r} KB" for r in pivot.index])
    ax.set_xlabel("PU fanout")
    ax.set_ylabel("AiM buffer")
    ax.set_title(title + ("  (log10)" if log else ""))
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(j, i, f"{v:.2e}", ha="center", va="center",
                    color="white", fontsize=8)
    plt.colorbar(im, ax=ax)


fig, axes = plt.subplots(2, 2, figsize=(14, 10))
heat(axes[0, 0], "total_area_mm2", "Total chip area (mm²)")
heat(axes[0, 1], "energy_j", "Workload energy (J)")
heat(axes[1, 0], "latency_s", "Workload latency (s)")
heat(axes[1, 1], "edp", "Energy·Delay product (J·s)")
plt.tight_layout()
plt.show()

## 5. Pareto view: area vs EDP

Each point is one (buffer, fanout) configuration. Points on the lower-left
frontier are the best area–EDP trade-offs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df["total_area_mm2"], df["edp"],
                c=df["pu_count"], s=60 + df["buffer_kb"] * 10,
                cmap="plasma", edgecolor="k")
for _, r in df.iterrows():
    ax.annotate(f"{int(r.buffer_kb)}KB,{int(r.pu_count)}",
                (r.total_area_mm2, r.edp),
                fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("Total chip area (mm²)")
ax.set_ylabel("Energy · Delay (J·s)")
ax.set_yscale("log")
ax.set_title("Area vs EDP  (size = buffer KB, color = PU count)")
plt.colorbar(sc, ax=ax, label="PU count")
plt.tight_layout()
plt.show()

## 6. Best configurations

Top 5 by each metric.

In [ ]:
for metric in ["energy_j", "latency_s", "edp", "total_area_mm2"]:
    print(f"\n== Top 5 by {metric} (lowest) ==")
    cols = ["buffer_kb", "pu_fanout", "pu_count",
            "total_area_mm2", "energy_j", "latency_s", "edp"]
    print(df.nsmallest(5, metric)[cols].to_string(index=False))